# Advanced Decorators and Descriptors — Tutorial-Style Problems with Solutions

This notebook is deliberately written as a **guided tutorial**.

Each advanced problem is broken into logical stages:

1. **Motivation**
2. **Mental model**
3. **Small implementation**
4. **Experiment**
5. **Edge case**
6. **Takeaway**

The focus is the same topic — **Python decorators and descriptors** — but the problems and explanations are new and more tutorial-oriented.

## Learning goals

You will practice:

- decorator factories,
- closure state,
- stacked decorators,
- metadata preservation,
- method decoration,
- async decorators,
- descriptor protocol tracing,
- data vs non-data descriptors,
- per-instance descriptor storage,
- reusable validators,
- normalization descriptors,
- weak-reference storage,
- lazy descriptors,
- controlled deletion,
- class decorators,
- transactional updates,
- combining decorators and descriptors into a mini framework.

In [57]:
from __future__ import annotations

import asyncio
import functools
import inspect
import time
from typing import Any, Callable
from weakref import WeakKeyDictionary

# Part I — Advanced Decorators

The most useful mental model is:

```python
@decorator
def f(...):
    ...
```

is approximately:

```python
f = decorator(f)
```

A decorator therefore changes the object bound to the function name.

## Problem 1 — Parameterized logging decorator

We want configuration at decoration time:

```python
@log_calls("DEBUG")
def compute(...):
    ...
```

That means `log_calls("DEBUG")` must return the real decorator.

### Step 1 — Recognize the three layers

A configurable decorator commonly has:

- a **factory** that receives configuration,
- a **decorator** that receives the function,
- a **wrapper** that receives call arguments.

Expanded:

```python
f = log_calls("DEBUG")(f)
```

In [58]:
def log_calls(level: str):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            print(f"[{level}] {fn.__qualname__}")
            return fn(*args, **kwargs)
        return wrapper
    return decorator

### Step 2 — Use the decorator

In [59]:
@log_calls("DEBUG")
def power(base, exponent=2):
    return base ** exponent

@log_calls("INFO")
def greet(name):
    return f"Hello, {name}!"

assert power(3, 3) == 27
assert greet("Ada") == "Hello, Ada!"

[DEBUG] power
[INFO] greet


### Step 3 — Verify introspection

`functools.wraps` preserves metadata and exposes `__wrapped__`.

In [60]:
assert power.__name__ == "power"
assert power.__wrapped__.__name__ == "power"
print(inspect.signature(power))

(base, exponent=2)


### Takeaway

A decorator with arguments is usually a **decorator factory**.

Always distinguish decoration-time arguments from call-time arguments.

## Problem 2 — Stateful decorator with a minimum call interval

Now the wrapper must remember information between calls.

We want each decorated function to have independent state.

### Step 1 — Choose state ownership

A global variable would mix unrelated functions.

A closure variable belongs naturally to one decorated function.

In [61]:
def minimum_interval(seconds: float):
    if seconds < 0:
        raise ValueError("seconds must be >= 0")

    def decorator(fn):
        last_called = None

        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            nonlocal last_called

            now = time.perf_counter()
            if last_called is not None:
                elapsed = now - last_called
                if elapsed < seconds:
                    time.sleep(seconds - elapsed)

            result = fn(*args, **kwargs)
            last_called = time.perf_counter()
            return result

        return wrapper
    return decorator

### Step 2 — Test it

In [62]:
timestamps = []

@minimum_interval(0.003)
def mark():
    timestamps.append(time.perf_counter())

mark()
mark()
mark()

assert timestamps[1] - timestamps[0] >= 0.002
assert timestamps[2] - timestamps[1] >= 0.002
print("gaps:", timestamps[1]-timestamps[0], timestamps[2]-timestamps[1])

gaps: 0.0032524000853300095 0.0031481999903917313


### Edge case

This version is intentionally single-threaded.

A production implementation used concurrently would need synchronization around shared wrapper state.

## Problem 3 — Validate return values

Instead of validating inputs, validate the function's result.

The decorator should not know the business rule itself; it receives a predicate.

In [63]:
def ensure_result(predicate: Callable[[Any], bool], message="invalid result"):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            result = fn(*args, **kwargs)
            if not predicate(result):
                raise ValueError(message)
            return result
        return wrapper
    return decorator

### Use one mechanism for different policies

In [64]:
@ensure_result(lambda x: x >= 0, "result must be non-negative")
def difference(a, b):
    return a - b

@ensure_result(lambda s: bool(s.strip()), "result cannot be blank")
def full_name(first, last):
    return f"{first} {last}"

assert difference(8, 3) == 5
assert full_name("Ada", "Lovelace") == "Ada Lovelace"

try:
    difference(2, 5)
except ValueError as exc:
    print(exc)

result must be non-negative


### Takeaway

Separate **mechanism** from **policy**:

- wrapper controls *when* validation happens,
- predicate defines *what* is valid.

## Problem 4 — Reject re-entrant calls

A re-entrant call enters a function again before the previous invocation has finished.

We will model the wrapper as a two-state machine.

### Step 1 — Why `finally` matters

The wrapper must clear its `running` flag even if the wrapped function raises.

Otherwise one failure could permanently lock the function.

In [65]:
def prevent_reentry(fn):
    running = False

    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        nonlocal running

        if running:
            raise RuntimeError(f"re-entrant call: {fn.__qualname__}")

        running = True
        try:
            return fn(*args, **kwargs)
        finally:
            running = False

    return wrapper

In [66]:
@prevent_reentry
def recurse(n):
    if n:
        return recurse(n - 1)
    return 0

try:
    recurse(1)
except RuntimeError as exc:
    print(exc)

re-entrant call: recurse


### Step 2 — Verify recovery after an ordinary exception

In [67]:
@prevent_reentry
def maybe_fail(flag):
    if flag:
        raise ValueError("boom")
    return "ok"

try:
    maybe_fail(True)
except ValueError:
    pass

assert maybe_fail(False) == "ok"

## Problem 5 — Reason about stacked decorators

For:

```python
@A
@B
@C
def task():
    ...
```

Python constructs:

```python
task = A(B(C(task)))
```

In [68]:
def tag(name):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            print(f"enter {name}")
            try:
                return fn(*args, **kwargs)
            finally:
                print(f"exit {name}")
        return wrapper
    return decorator

### Predict the output before running

In [69]:
@tag("A")
@tag("B")
@tag("C")
def stacked():
    print("body")

stacked()

enter A
enter B
enter C
body
exit C
exit B
exit A


### Expected nesting

```text
enter A
enter B
enter C
body
exit C
exit B
exit A
```

Decorator order matters whenever wrappers have side effects or state.

## Problem 6 — Why decorated instance methods still bind `self`

Python functions are descriptors.

If a function decorator returns another normal function, that wrapper also participates in method binding.

In [70]:
def announce(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        print(f"running {fn.__qualname__}")
        return fn(*args, **kwargs)
    return wrapper

class Counter:
    def __init__(self, value=0):
        self.value = value

    @announce
    def increment(self, amount=1):
        self.value += amount
        return self.value

c = Counter(10)
assert c.increment(5) == 15

running Counter.increment


### Takeaway

The replacement wrapper is itself a function descriptor.

That is why `c.increment(...)` still becomes a bound method and receives `c` as its first argument.

## Problem 7 — Async timeout decorator

An async-aware wrapper should itself be `async def` and should `await` the original coroutine.

We will use `asyncio.wait_for`.

In [71]:
def async_timeout(seconds: float):
    if seconds <= 0:
        raise ValueError("seconds must be > 0")

    def decorator(fn):
        if not inspect.iscoroutinefunction(fn):
            raise TypeError("async_timeout requires async def")

        @functools.wraps(fn)
        async def wrapper(*args, **kwargs):
            return await asyncio.wait_for(
                fn(*args, **kwargs),
                timeout=seconds,
            )
        return wrapper

    return decorator

### Fast coroutine

In [72]:
@async_timeout(0.1)
async def fast_operation():
    await asyncio.sleep(0)
    return "done"

assert await fast_operation() == "done"

### Slow coroutine

In [73]:
@async_timeout(0.003)
async def slow_operation():
    await asyncio.sleep(0.02)
    return "late"

try:
    await slow_operation()
except asyncio.TimeoutError:
    print("timeout as expected")

timeout as expected


### Takeaway

A decorator should preserve the target function's execution model.

For `async def`, that means returning an async wrapper when asynchronous control is required.

# Part II — Advanced Descriptors

A descriptor is an object stored on a class that implements one or more of:

- `__get__`
- `__set__`
- `__delete__`

Descriptors let class attributes actively participate in attribute access.

## Problem 8 — Trace the descriptor protocol

Before building abstractions, make the protocol visible.

In [74]:
class VerboseDescriptor:
    def __set_name__(self, owner, name):
        self.name = name
        print(f"__set_name__ -> {owner.__name__}.{name}")

    def __get__(self, instance, owner=None):
        print(f"__get__ -> instance={instance!r}, owner={owner.__name__}")
        if instance is None:
            return self
        return instance.__dict__.get(self.name)

    def __set__(self, instance, value):
        print(f"__set__ -> {value!r}")
        instance.__dict__[self.name] = value

class Demo:
    value = VerboseDescriptor()

__set_name__ -> Demo.value


### Assignment

Because this descriptor defines `__set__`, normal assignment is intercepted.

In [75]:
d = Demo()
d.value = 10
assert d.__dict__["value"] == 10

__set__ -> 10


### Instance read

In [76]:
assert d.value == 10

__get__ -> instance=<__main__.Demo object at 0x000002117902D940>, owner=Demo


### Class read

For `Demo.value`, `instance` is `None`.

Returning the descriptor itself is the conventional class-access behavior.

In [77]:
assert isinstance(Demo.value, VerboseDescriptor)

__get__ -> instance=None, owner=Demo


## Problem 9 — Diagnose the shared descriptor-state bug

The descriptor object itself is shared by all instances.

Therefore storing normal field data on `self` inside the descriptor is usually wrong.

In [78]:
class BadField:
    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return self.value

    def __set__(self, instance, value):
        self.value = value

class BadPoint:
    x = BadField()

p1 = BadPoint()
p2 = BadPoint()

p1.x = 10
p2.x = 99

assert p1.x == 99
assert p2.x == 99
print("shared result:", p1.x, p2.x)

shared result: 99 99


### Fix: store values on each instance

In [79]:
class InstanceField:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        instance.__dict__[self.name] = value

class GoodPoint:
    x = InstanceField()

a = GoodPoint()
b = GoodPoint()

a.x = 10
b.x = 99

assert a.x == 10
assert b.x == 99

### Takeaway

The descriptor is shared.

The managed field value usually belongs to the instance.

## Problem 10 — Data vs non-data descriptor precedence

A non-data descriptor defines `__get__` only.

A data descriptor also defines `__set__` or `__delete__`.

That difference changes lookup order.

In [80]:
class NonData:
    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return "non-data descriptor"

class Data:
    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return "data descriptor"

    def __set__(self, instance, value):
        instance.__dict__["_stored"] = value

class LookupDemo:
    non_data = NonData()
    data = Data()

### Force name collisions in `__dict__`

In [81]:
obj = LookupDemo()
obj.__dict__["non_data"] = "instance value"
obj.__dict__["data"] = "instance value"

assert obj.non_data == "instance value"
assert obj.data == "data descriptor"

print(obj.non_data)
print(obj.data)

instance value
data descriptor


### Simplified lookup model

For `obj.attr`:

1. data descriptor,
2. instance dictionary,
3. non-data descriptor / class attribute,
4. `__getattr__` fallback.

This rule explains many descriptor patterns.

## Problem 11 — Reusable validation descriptors

We want subclasses to define validation rules without rewriting storage mechanics.

In [82]:
class Validated:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        if self.name not in instance.__dict__:
            raise AttributeError(f"{self.name} has not been assigned")
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        self.validate(value)
        instance.__dict__[self.name] = value

    def validate(self, value):
        raise NotImplementedError

### Text validator

In [83]:
class Text(Validated):
    def __init__(self, *, min_length=0, max_length=None):
        self.min_length = min_length
        self.max_length = max_length

    def validate(self, value):
        if not isinstance(value, str):
            raise TypeError(f"{self.name} must be a string")
        if len(value) < self.min_length:
            raise ValueError(f"{self.name} too short")
        if self.max_length is not None and len(value) > self.max_length:
            raise ValueError(f"{self.name} too long")

### Integer validator

`bool` is a subclass of `int`, so reject it explicitly for strict integer fields.

In [84]:
class Integer(Validated):
    def __init__(self, *, minimum=None, maximum=None):
        self.minimum = minimum
        self.maximum = maximum

    def validate(self, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError(f"{self.name} must be an integer")
        if self.minimum is not None and value < self.minimum:
            raise ValueError(f"{self.name} must be >= {self.minimum}")
        if self.maximum is not None and value > self.maximum:
            raise ValueError(f"{self.name} must be <= {self.maximum}")

### Use the hierarchy

In [85]:
class Person:
    name = Text(min_length=1, max_length=50)
    age = Integer(minimum=0, maximum=130)

    def __init__(self, name, age):
        self.name = name
        self.age = age

person = Person("Ada", 36)
assert person.name == "Ada"
assert person.age == 36

for bad in (-1, 999, True):
    try:
        person.age = bad
    except (TypeError, ValueError) as exc:
        print(type(exc).__name__, exc)

ValueError age must be >= 0
ValueError age must be <= 130
TypeError age must be an integer


## Problem 12 — Normalization instead of rejection

Descriptors can transform data before storage.

Build a reusable text field that trims whitespace and optionally lowercases.

In [86]:
class NormalizedText:
    def __init__(self, *, lowercase=False):
        self.lowercase = lowercase

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        if not isinstance(value, str):
            raise TypeError(f"{self.name} must be a string")

        value = value.strip()
        if self.lowercase:
            value = value.lower()

        instance.__dict__[self.name] = value

In [87]:
class Customer:
    name = NormalizedText()
    email = NormalizedText(lowercase=True)

customer = Customer()
customer.name = "  Ada Lovelace "
customer.email = " ADA@EXAMPLE.COM "

assert customer.name == "Ada Lovelace"
assert customer.email == "ada@example.com"
print(customer.__dict__)

{'name': 'Ada Lovelace', 'email': 'ada@example.com'}


### Takeaway

Descriptors can centralize:

- validation,
- normalization,
- conversion,
- auditing,
- access restrictions,
- lazy loading.

The public interface can remain ordinary attribute syntax.

## Problem 13 — Store values outside the instance dictionary

Use `WeakKeyDictionary` to keep per-instance values in the descriptor without unnecessarily keeping instances alive.

In [88]:
class ExternalStorage:
    def __init__(self):
        self._values = WeakKeyDictionary()

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        try:
            return self._values[instance]
        except KeyError:
            raise AttributeError(f"{self.name} is unset") from None

    def __set__(self, instance, value):
        self._values[instance] = value

In [89]:
class Session:
    token = ExternalStorage()

s1 = Session()
s2 = Session()

s1.token = "alpha"
s2.token = "beta"

assert s1.token == "alpha"
assert s2.token == "beta"
assert s1.__dict__ == {}
assert s2.__dict__ == {}

print(s1.__dict__, s2.__dict__)

{} {}


### Design note

Changing the storage strategy can also change object requirements.

Weak-key storage is useful, but it is not universally interchangeable with `instance.__dict__`.

## Problem 14 — Lazy value as a non-data descriptor

We want the first access to compute a value and later accesses to come directly from `instance.__dict__`.

That is a natural job for a non-data descriptor.

In [90]:
class lazy_value:
    def __init__(self, fn):
        self.fn = fn
        functools.update_wrapper(self, fn)

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self

        value = self.fn(instance)
        instance.__dict__[self.name] = value
        return value

### Observe one computation

In [91]:
class Statistics:
    def __init__(self, values):
        self.values = values
        self.compute_count = 0

    @lazy_value
    def average(self):
        self.compute_count += 1
        return sum(self.values) / len(self.values)

stats = Statistics([10, 20, 30])

assert stats.average == 20
assert stats.compute_count == 1
assert stats.average == 20
assert stats.compute_count == 1

print(stats.__dict__)

{'values': [10, 20, 30], 'compute_count': 1, 'average': 20.0}


### Why no `__set__`?

Adding `__set__` would turn the descriptor into a data descriptor.

Then it would outrank the instance dictionary, defeating this same-name shadowing technique.

## Problem 15 — Controlled deletion with `__delete__`

Descriptors can decide whether `del obj.attr` is allowed.

In [92]:
class DeletableField:
    def __init__(self, *, allow_delete=False):
        self.allow_delete = allow_delete

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        if self.name not in instance.__dict__:
            raise AttributeError(self.name)
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        instance.__dict__[self.name] = value

    def __delete__(self, instance):
        if not self.allow_delete:
            raise AttributeError(f"{self.name} cannot be deleted")
        try:
            del instance.__dict__[self.name]
        except KeyError:
            raise AttributeError(self.name) from None

In [93]:
class Document:
    title = DeletableField()
    scratch = DeletableField(allow_delete=True)

doc = Document()
doc.title = "Metaprogramming"
doc.scratch = "draft"

del doc.scratch

try:
    doc.scratch
except AttributeError:
    print("scratch deleted")

try:
    del doc.title
except AttributeError as exc:
    print(exc)

scratch deleted
title cannot be deleted


# Part III — Combining Both Techniques

## Problem 16 — Audited descriptor assignments

The descriptor should record every successful change without requiring the model class to contain audit code.

In [94]:
_MISSING = object()

class AuditedField:
    def __init__(self, validator=None):
        self.validator = validator

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        if self.validator is not None:
            self.validator(value)

        old = instance.__dict__.get(self.name, _MISSING)
        instance.__dict__[self.name] = value

        instance.__dict__.setdefault("_audit", []).append({
            "field": self.name,
            "old": "<MISSING>" if old is _MISSING else old,
            "new": value,
        })

### Supply policy functions

In [95]:
def non_blank(value):
    if not isinstance(value, str):
        raise TypeError("must be a string")
    if not value.strip():
        raise ValueError("cannot be blank")

def non_negative(value):
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise TypeError("must be numeric")
    if value < 0:
        raise ValueError("cannot be negative")

In [96]:
class Product:
    name = AuditedField(non_blank)
    price = AuditedField(non_negative)

    def __init__(self, name, price):
        self.name = name
        self.price = price

product = Product("Keyboard", 100)
product.price = 90

assert product._audit[-1] == {
    "field": "price",
    "old": 100,
    "new": 90,
}

for event in product._audit:
    print(event)

{'field': 'name', 'old': '<MISSING>', 'new': 'Keyboard'}
{'field': 'price', 'old': '<MISSING>', 'new': 100}
{'field': 'price', 'old': 100, 'new': 90}


## Problem 17 — Discover descriptor fields with a class decorator

A class decorator can scan the class hierarchy and build schema metadata.

In [97]:
def collect_fields(cls):
    fields = {}

    for base in reversed(cls.__mro__):
        for name, value in vars(base).items():
            if isinstance(value, Validated):
                fields[name] = value

    cls.__fields__ = fields
    return cls

In [98]:
@collect_fields
class Employee:
    name = Text(min_length=1)
    age = Integer(minimum=18)

    def __init__(self, name, age):
        self.name = name
        self.age = age

assert set(Employee.__fields__) == {"name", "age"}
assert Employee.__fields__["name"] is Employee.name
print(Employee.__fields__)

{'name': <__main__.Text object at 0x0000021178FC5310>, 'age': <__main__.Integer object at 0x0000021178FC5450>}


### Framework pattern

Descriptors declare behavior.

A class decorator discovers those descriptors and creates metadata.

That is a lightweight declarative framework.

## Problem 18 — Transactional method decorator

If a method mutates state and then fails, restore selected attributes automatically.

In [99]:
def rollback_fields(*names):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(self, *args, **kwargs):
            snapshot = {name: getattr(self, name) for name in names}

            try:
                return fn(self, *args, **kwargs)
            except Exception:
                for name, old_value in snapshot.items():
                    setattr(self, name, old_value)
                raise

        return wrapper
    return decorator

In [100]:
class Account:
    balance = Integer(minimum=0)

    def __init__(self, balance):
        self.balance = balance

    @rollback_fields("balance")
    def charge(self, amount, *, fail=False):
        self.balance -= amount
        if fail:
            raise RuntimeError("external failure")
        return self.balance

account = Account(100)
assert account.charge(25) == 75

try:
    account.charge(20, fail=True)
except RuntimeError:
    pass

assert account.balance == 75
print("balance after rollback:", account.balance)

balance after rollback: 75


# Capstone — Tiny Declarative Model Framework

Build a small framework from the previous ideas.

Features:

- descriptor fields,
- validation,
- automatic field discovery,
- serialization,
- transactional bulk updates,
- public-method auditing.

## Step 1 — Base model field

In [101]:
class ModelField:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        if self.name not in instance.__dict__:
            raise AttributeError(f"{self.name} is unset")
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        self.validate(value)
        instance.__dict__[self.name] = value

    def validate(self, value):
        raise NotImplementedError

## Step 2 — Concrete fields

In [102]:
class ModelText(ModelField):
    def __init__(self, *, min_length=0):
        self.min_length = min_length

    def validate(self, value):
        if not isinstance(value, str):
            raise TypeError(f"{self.name} must be a string")
        if len(value) < self.min_length:
            raise ValueError(f"{self.name} is too short")

class ModelInteger(ModelField):
    def __init__(self, *, minimum=None, maximum=None):
        self.minimum = minimum
        self.maximum = maximum

    def validate(self, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError(f"{self.name} must be an integer")
        if self.minimum is not None and value < self.minimum:
            raise ValueError(f"{self.name} must be >= {self.minimum}")
        if self.maximum is not None and value > self.maximum:
            raise ValueError(f"{self.name} must be <= {self.maximum}")

## Step 3 — Model class decorator

In [103]:
def model(cls):
    fields = {}
    for base in reversed(cls.__mro__):
        for name, value in vars(base).items():
            if isinstance(value, ModelField):
                fields[name] = value
    cls.__fields__ = fields
    return cls

## Step 4 — Audit public methods

In [104]:
def audit_public_methods(cls):
    for name, value in list(vars(cls).items()):
        if name.startswith("_") or not inspect.isfunction(value):
            continue

        def make_wrapper(fn):
            @functools.wraps(fn)
            def wrapper(self, *args, **kwargs):
                self.__dict__.setdefault("_method_log", []).append({
                    "method": fn.__name__,
                    "args": args,
                    "kwargs": dict(kwargs),
                })
                return fn(self, *args, **kwargs)
            return wrapper

        setattr(cls, name, make_wrapper(value))

    return cls

## Step 5 — Base model operations

In [105]:
class BaseModel:
    def as_dict(self):
        return {name: getattr(self, name) for name in self.__fields__}

    def update(self, **changes):
        unknown = set(changes) - set(self.__fields__)
        if unknown:
            raise AttributeError(
                "unknown field(s): " + ", ".join(sorted(unknown))
            )

        snapshot = self.as_dict()

        try:
            for name, value in changes.items():
                setattr(self, name, value)
        except Exception:
            for name, old_value in snapshot.items():
                setattr(self, name, old_value)
            raise

        return self

## Step 6 — Define a model declaratively

In [106]:
@model
@audit_public_methods
class User(BaseModel):
    username = ModelText(min_length=3)
    age = ModelInteger(minimum=0, maximum=130)

    def __init__(self, username, age):
        self.username = username
        self.age = age

    def birthday(self):
        self.age += 1
        return self.age

## Step 7 — Exercise the framework

In [107]:
user = User("ada", 36)

assert user.as_dict() == {"username": "ada", "age": 36}
assert user.birthday() == 37
assert user._method_log[-1]["method"] == "birthday"

print(user.as_dict())
print(user._method_log)

{'username': 'ada', 'age': 37}
[{'method': 'birthday', 'args': (), 'kwargs': {}}]


## Step 8 — Prove rollback works

In [108]:
before = user.as_dict()

try:
    user.update(username="grace", age=500)
except ValueError as exc:
    print("rejected:", exc)

assert user.as_dict() == before
print("after failed update:", user.as_dict())

rejected: age must be <= 130
after failed update: {'username': 'ada', 'age': 37}


## Step 9 — Inspect descriptor configuration through the class

In [109]:
assert isinstance(User.username, ModelText)
assert isinstance(User.age, ModelInteger)

print(User.__fields__)
print("username min length:", User.username.min_length)
print("age range:", User.age.minimum, User.age.maximum)

{'username': <__main__.ModelText object at 0x000002117902CEC0>, 'age': <__main__.ModelInteger object at 0x000002117902F380>}
username min length: 3
age range: 0 130


# Extra Guided Problems

## Extra A — Write-once descriptor

Allow the first assignment but reject subsequent assignments.

In [110]:
class WriteOnce:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        if self.name in instance.__dict__:
            raise AttributeError(f"{self.name} is write-once")
        instance.__dict__[self.name] = value

class Record:
    identifier = WriteOnce()

r = Record()
r.identifier = 100
assert r.identifier == 100

try:
    r.identifier = 200
except AttributeError as exc:
    print(exc)

identifier is write-once


## Extra B — Count method calls across all instances

Store the counter on the wrapper itself.

In [111]:
def count_method_calls(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return fn(*args, **kwargs)

    wrapper.calls = 0
    return wrapper

class Worker:
    @count_method_calls
    def work(self):
        return "done"

w1 = Worker()
w2 = Worker()

w1.work()
w2.work()
w2.work()

assert Worker.work.calls == 3
print("shared call count:", Worker.work.calls)

shared call count: 3


## Extra C — Count calls per instance

Move state ownership from the wrapper to each instance.

In [112]:
def count_calls_per_instance(fn):
    counter_name = f"_{fn.__name__}_calls"

    @functools.wraps(fn)
    def wrapper(self, *args, **kwargs):
        setattr(
            self,
            counter_name,
            getattr(self, counter_name, 0) + 1,
        )
        return fn(self, *args, **kwargs)

    return wrapper

class PerInstanceWorker:
    @count_calls_per_instance
    def work(self):
        return "done"

wa = PerInstanceWorker()
wb = PerInstanceWorker()

wa.work()
wa.work()
wb.work()

assert wa._work_calls == 2
assert wb._work_calls == 1
print(wa.__dict__, wb.__dict__)

{'_work_calls': 2} {'_work_calls': 1}


# Final Review

## Decorators

Use decorators to transform callable or class behavior at definition time.

Typical uses:

- logging,
- retries,
- caching,
- timing,
- authorization,
- transactions,
- validation,
- async policies.

## Descriptors

Use descriptors to centralize attribute access behavior.

Typical uses:

- validation,
- conversion,
- normalization,
- lazy loading,
- auditing,
- read/write restrictions,
- external storage.

## The deeper connection

Python functions are descriptors too.

That is why methods bind automatically to instances.

Decorators and descriptors are both parts of Python's broader metaprogramming model.

# Further Challenges

Try these next:

1. cache only successful function results,
2. one decorator supporting both sync and async callables,
3. descriptor default values,
4. automatic type conversion,
5. schema aliases,
6. generated `__repr__`,
7. inherited field overrides,
8. descriptor change listeners,
9. expiring cached descriptors,
10. auto-generated `__init__` from field declarations.